# DLAI Model Merging - Scope-density TIES

Uniform early-layer attenuation did not help. This experiment instead changes how selectively TIES trims individual coordinates in embeddings, early blocks, and late blocks. Seed 42 selects one global schedule across all six pairs; seeds 7 and 123 are held out for the final verdict.

Use **Add Input** to attach notebook 02's Output containing `pilot_specialists_seed42.zip`. Select **GPU T4 x2**, enable Internet, and Run All.

In [ ]:
!nvidia-smi
!find /kaggle/input -maxdepth 3 -type f | head -50

## Install project and load seed-42 development specialists

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
REPO='https://github.com/LeuxLello/Dlai-model-merging.git'; BRANCH='codex/multiseed-results'
WORKDIR=Path('/kaggle/working/Dlai-model-merging')
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO,str(WORKDIR)])
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',str(WORKDIR)])
sys.path.insert(0,str(WORKDIR/'src')); os.chdir(WORKDIR)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(); print('Commit:',COMMIT)
bundles=list(Path('/kaggle/input').rglob('pilot_specialists_seed42.zip'))
assert bundles, 'Attach notebook-02 Output containing pilot_specialists_seed42.zip.'
DEV_ROOT=Path('/kaggle/working/pilot_specialists_seed42')
if DEV_ROOT.exists(): shutil.rmtree(DEV_ROOT)
with zipfile.ZipFile(bundles[0]) as archive: archive.extractall(DEV_ROOT)
assert len(list(DEV_ROOT.rglob('encoder.pt')))==4

## Environment, scopes, and pre-declared schedules

In [ ]:
import gc, itertools, json, platform
import numpy as np, pandas as pd, torch
from transformers import AutoModelForSequenceClassification
from dlai_merge.ablation import bert_mini_scopes
from dlai_merge.evaluation import TaskEvaluator
from dlai_merge.merging import mean_merge, ties_merge, ties_merge_by_scope
from dlai_merge.training import TrainConfig, train_specialist
assert torch.cuda.is_available(), 'Enable GPU T4 x2.'
GPU=torch.cuda.get_device_name(0); CAP=torch.cuda.get_device_capability(0)
assert CAP[0]>=7, 'Use GPU T4 x2, not P100.'; torch.ones(1,device='cuda').add_(1)
TASKS=['sst2','imdb','mrpc','rte']; PAIRS=list(itertools.combinations(TASKS,2))
BASE='prajjwal1/bert-mini'; DEV_SEED=42; HELDOUT_SEEDS=[7,123]
SCHEDULES={
 'uniform':{'embeddings':0.20,'early':0.20,'late':0.20},
 'early_sparse':{'embeddings':0.20,'early':0.10,'late':0.20},
 'early_very_sparse':{'embeddings':0.20,'early':0.05,'late':0.20},
 'depth_progressive':{'embeddings':0.10,'early':0.10,'late':0.30}}
base_model=AutoModelForSequenceClassification.from_pretrained(BASE,num_labels=2)
base_encoder={k:v.detach().cpu().clone() for k,v in base_model.base_model.state_dict().items()}
all_scopes=bert_mini_scopes(base_encoder.keys())
scopes={name:all_scopes[name] for name in ['embeddings','early','late']}
flat=[k for keys in scopes.values() for k in keys]
assert len(flat)==len(set(flat))==len(base_encoder) and set(flat)==set(base_encoder)
def scoped_ties(states,schedule):
    return ties_merge_by_scope(base_encoder,states,{name:(scopes[name],density) for name,density in schedule.items()})
def evaluate(rows,evaluators,references,seed,pair,method,schedule,state,phase):
    for task in pair:
        result=evaluators[task].evaluate(state); score=result['primary_score']
        rows.append({'phase':phase,'seed':seed,'pair':'+'.join(pair),'task':task,'method':method,'schedule':schedule,
          'score':score,'specialist_score':references[task],'retained':score/references[task]})
print('GPU:',GPU,'| scopes:',{k:len(v) for k,v in scopes.items()},'| schedules:',SCHEDULES)

## Development selection on seed 42
The winner maximizes average retention over all pair-task observations; worst retention and then the no-change control order break ties. No pair-specific tuning is performed.

In [ ]:
def artifact(task,name):
    found=list(DEV_ROOT.rglob(f'{task}/seed-{DEV_SEED}/{name}')); assert len(found)==1; return found[0]
dev_encoders={t:torch.load(artifact(t,'encoder.pt'),map_location='cpu',weights_only=True) for t in TASKS}
dev_heads={t:torch.load(artifact(t,'head.pt'),map_location='cpu',weights_only=True) for t in TASKS}
dev_eval={t:TaskEvaluator(t,dev_heads[t],max_eval_samples=2000,seed=DEV_SEED,output_root='/kaggle/working/scope-dev-eval') for t in TASKS}
dev_ref={t:dev_eval[t].evaluate(dev_encoders[t])['primary_score'] for t in TASKS}
dev_rows=[]
for pair in PAIRS:
    states=[dev_encoders[t] for t in pair]
    for name,schedule in SCHEDULES.items(): evaluate(dev_rows,dev_eval,dev_ref,DEV_SEED,pair,'scope_ties',name,scoped_ties(states,schedule),'development')
development_results=pd.DataFrame(dev_rows); assert len(development_results)==48
development_selection=(development_results.groupby('schedule',as_index=False).agg(mean_retained=('retained','mean'),worst_retained=('retained','min')))
order={name:i for i,name in enumerate(SCHEDULES)}; development_selection['order']=development_selection.schedule.map(order)
ranked=development_selection.sort_values(['mean_retained','worst_retained','order'],ascending=[False,False,True])
SELECTED=str(ranked.iloc[0].schedule); print('Frozen held-out schedule:',SELECTED,SCHEDULES[SELECTED]); development_selection

## Held-out confirmation on seeds 7 and 123

In [ ]:
TRAIN_ROOT=Path('/kaggle/working/scope_density_specialists'); heldout_rows=[]; specialist_rows=[]
for seed in HELDOUT_SEEDS:
    print('\n===== HELD-OUT SEED',seed,'=====')
    for task in TASKS:
        summary=train_specialist(TrainConfig(task=task,output_root=str(TRAIN_ROOT),seed=seed,max_train_samples=12000,max_eval_samples=2000,epochs=3,max_steps=400,eval_steps=100,train_batch_size=32,eval_batch_size=64,learning_rate=2e-5))
        metric=summary['primary_metric']; specialist_rows.append({'seed':seed,'task':task,'primary_metric':metric,'score':summary['eval_metrics']['eval_'+metric]})
    enc={t:torch.load(TRAIN_ROOT/t/f'seed-{seed}'/'encoder.pt',map_location='cpu',weights_only=True) for t in TASKS}
    heads={t:torch.load(TRAIN_ROOT/t/f'seed-{seed}'/'head.pt',map_location='cpu',weights_only=True) for t in TASKS}
    evaluators={t:TaskEvaluator(t,heads[t],max_eval_samples=2000,seed=seed,output_root=f'/kaggle/working/scope-heldout-{seed}') for t in TASKS}
    references={t:evaluators[t].evaluate(enc[t])['primary_score'] for t in TASKS}
    for pair in PAIRS:
        states=[enc[t] for t in pair]
        candidates={'mean':('not_applicable',mean_merge(base_encoder,states)),
          'ties':('uniform',ties_merge(base_encoder,states,density=0.2,scale=1.0)),
          'scope_ties':(SELECTED,scoped_ties(states,SCHEDULES[SELECTED]))}
        for method,(schedule,state) in candidates.items(): evaluate(heldout_rows,evaluators,references,seed,pair,method,schedule,state,'held_out')
    for task in TASKS: shutil.rmtree(TRAIN_ROOT/task/f'seed-{seed}'/'trainer',ignore_errors=True)
    del enc,heads,evaluators; gc.collect(); torch.cuda.empty_cache()
heldout_results=pd.DataFrame(heldout_rows); assert len(heldout_results)==72
print('Held-out task-level rows:',len(heldout_results))

## Primary paired analysis and bootstrap interval

In [ ]:
per_pair=(heldout_results.groupby(['seed','pair','method','schedule'],as_index=False).agg(mean_retained=('retained','mean'),worst_retained=('retained','min')))
method_summary=(per_pair.groupby('method',as_index=False).agg(mean_retained=('mean_retained','mean'),sd=('mean_retained','std'),worst_retained=('worst_retained','min')))
a=per_pair[per_pair.method=='scope_ties'][['seed','pair','mean_retained']].rename(columns={'mean_retained':'scope_ties'})
b=per_pair[per_pair.method=='ties'][['seed','pair','mean_retained']].rename(columns={'mean_retained':'ties'})
paired=a.merge(b,on=['seed','pair']); paired['delta']=paired.scope_ties-paired.ties
delta=paired.delta.to_numpy(); rng=np.random.default_rng(2026); boot=delta[rng.integers(0,len(delta),size=(10000,len(delta)))].mean(axis=1)
improvement=pd.DataFrame([{'selected_schedule':SELECTED,'n_pair_seed_units':len(delta),'mean_delta':delta.mean(),'median_delta':np.median(delta),'win_rate':(delta>0).mean(),'tie_rate':np.isclose(delta,0,atol=1e-12).mean(),'worst_delta':delta.min(),'best_delta':delta.max(),'bootstrap_ci_low':np.quantile(boot,0.025),'bootstrap_ci_high':np.quantile(boot,0.975)}])
display(method_summary); display(improvement); display(paired)

## Figures and reproducible export

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
FIG=Path('/kaggle/working/scope_density_figures'); FIG.mkdir(exist_ok=True)
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
sns.barplot(data=development_selection.sort_values('order'),x='schedule',y='mean_retained',ax=axes[0]); axes[0].tick_params(axis='x',rotation=20); axes[0].set_title('Development schedule selection')
sns.stripplot(data=paired,x='pair',y='delta',hue='seed',dodge=True,ax=axes[1]); axes[1].axhline(0,color='black',lw=1); axes[1].tick_params(axis='x',rotation=25); axes[1].set_title('Held-out scope-TIES minus TIES')
fig.tight_layout(); figure=FIG/'scope_density_ties.png'; fig.savefig(figure,dpi=180,bbox_inches='tight'); plt.show()
OUT=Path('/kaggle/working/scope_density_ties_results'); OUT.mkdir(exist_ok=True)
development_results.to_csv(OUT/'development_results.csv',index=False); development_selection.to_csv(OUT/'development_selection.csv',index=False)
heldout_results.to_csv(OUT/'heldout_results.csv',index=False); per_pair.to_csv(OUT/'heldout_per_pair.csv',index=False)
pd.DataFrame(specialist_rows).to_csv(OUT/'heldout_specialists.csv',index=False); method_summary.to_csv(OUT/'method_summary.csv',index=False); paired.to_csv(OUT/'paired_differences.csv',index=False); improvement.to_csv(OUT/'improvement_summary.csv',index=False)
metadata={'purpose':'scope-density TIES development and held-out confirmation','commit':COMMIT,'base_model':BASE,'tasks':TASKS,'development_seed':DEV_SEED,'heldout_seeds':HELDOUT_SEEDS,'candidate_schedules':SCHEDULES,'selected_schedule':SELECTED,'gpu':GPU,'python':platform.python_version(),'torch':torch.__version__,'primary_claim_uses_heldout_only':True,'bootstrap_resamples':10000}
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2)); shutil.copy2(figure,OUT/figure.name)
archive=shutil.make_archive('/kaggle/working/scope_density_ties_results','zip',OUT); print(archive)